# Mixed Precision Training

Automatic mixed precision (AMP) uses lower-precision operations where they are safe while retaining higher precision where it is needed. On supported accelerators this can reduce memory use and improve training throughput.


In [ ]:
from time import perf_counter

import torch
from torch import nn

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autocast_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16

print(f"Device: {device}")
print(f"Autocast dtype: {autocast_dtype}")


## Prepare the Model and Batch

The same model architecture and batch will be used for standard and mixed-precision steps. A fixed seed makes the setup reproducible.


In [ ]:
def create_model_and_optimizer():
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(128, 256),
        nn.ReLU(),
        nn.Linear(256, 10),
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    return model, optimizer

features = torch.randn(512, 128, device=device)
targets = torch.randint(0, 10, (512,), device=device)
loss_fn = nn.CrossEntropyLoss()


## Standard Precision Step

The standard training order is clear gradients, run the forward pass, calculate loss, call `backward()`, and update the optimizer.


In [ ]:
standard_model, standard_optimizer = create_model_and_optimizer()
standard_model.train()
standard_optimizer.zero_grad(set_to_none=True)

standard_start = perf_counter()
standard_logits = standard_model(features)
standard_loss = loss_fn(standard_logits, targets)
standard_loss.backward()
standard_optimizer.step()
if device.type == "cuda":
    torch.cuda.synchronize()
standard_elapsed = perf_counter() - standard_start

print(f"Standard loss: {standard_loss.item():.4f}")
print(f"Standard step: {standard_elapsed:.6f} seconds")


## Autocast and Gradient Scaling

`autocast` selects an appropriate dtype for each operation. CUDA float16 gradients can underflow, so `GradScaler` scales the loss before `backward()` and safely unscales gradients during the optimizer step. CPU bfloat16 has a wider exponent range and does not need scaling.


In [ ]:
amp_model, amp_optimizer = create_model_and_optimizer()
amp_model.train()

if hasattr(torch.amp, "GradScaler"):
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
else:
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

amp_optimizer.zero_grad(set_to_none=True)
amp_start = perf_counter()
with torch.amp.autocast(device_type=device.type, dtype=autocast_dtype):
    amp_logits = amp_model(features)
    amp_loss = loss_fn(amp_logits, targets)

scaler.scale(amp_loss).backward()
scaler.step(amp_optimizer)
scaler.update()
if device.type == "cuda":
    torch.cuda.synchronize()
amp_elapsed = perf_counter() - amp_start

print(f"AMP loss: {amp_loss.item():.4f}")
print(f"GradScaler enabled: {scaler.is_enabled()}")
print(f"AMP step: {amp_elapsed:.6f} seconds")


## Compare the Steps

A single timed step demonstrates the APIs but is not a valid performance benchmark. For a real comparison, warm up the device, run many steps, synchronize CUDA before reading the timer, and measure peak memory as well as elapsed time.


In [ ]:
print(f"Standard precision: {standard_elapsed:.6f} seconds")
print(f"Mixed precision:    {amp_elapsed:.6f} seconds")

if device.type == "cuda":
    print(f"Current CUDA memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MiB")
else:
    print("Run on CUDA to observe the usual AMP speed and memory benefits.")


## PTCA Review

Set the model to training mode before continuing training. Keep the forward pass and loss inside `autocast`. Keep `backward()` outside it. When gradient scaling is enabled, call `scaler.scale(loss).backward()`, `scaler.step(optimizer)`, and `scaler.update()` in that order.
